# Этап 1 — Разведочный анализ (EDA)

**Датасет:** Online Retail II (UCI) — транзакции британского интернет-магазина
за 2009–2010 гг., ~525 тыс. строк. Один ряд = одна позиция в чеке
(товар, количество, цена, дата, клиент, страна).

**Цель ноутбука:** изучить данные, найти проблемные строки и принять обоснованные
решения по очистке, чтобы на этапе 2 корректно посчитать RFM-метрики
(Recency / Frequency / Monetary) на каждого клиента.

**План:** обзор структуры → поиск проблем → анализ их природы и пересечений
→ решения по очистке.

In [51]:
# импорт нужных библиотек
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [52]:
# загрузка базы и проверка ее корректности
df = pd.read_excel("../data/raw/online_retail_II.xlsx")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [ ]:
# описание таблицы
# в двух столбцах есть отсутствующие данные
df.info()

In [ ]:
# количество пустых значений
df.isnull().sum()

Invoice             0
StockCode           0
Description      2928
Quantity            0
InvoiceDate         0
Price               0
Customer ID    107927
Country             0
dtype: int64

In [ ]:
# начинаю смотреть таблицы с проблемными строками, в том числе с отсутствующими данными

# просмотр таблицы с отсутствующими данными в столбце 'Customer ID'
# смущает отрицательный 'Quantity' и 'Price'= 0
print('=== Пропуски в Customer ID ===')
display(df[df['Customer ID'].isna()].sample(10, random_state=50))

# просмотр таблицы с отрицательными данными в колонке 'Quantity'
print('=== Отрицательные Quantity ===')
display(df[df['Quantity'] < 0].sample(10, random_state=50))

# просмотр таблицы со столбцом 'Price'= 0
print('=== Price равный 0 ===')
display(df[df['Price'] == 0].sample(10, random_state=50))

# просмотр таблицы с отсутствующими данными в столбце 'Description'
print('=== Пропуски в Description ===')
display(df[df['Description'].isna()].sample(10, random_state=60))

In [ ]:
no_id    = df['Customer ID'].isna()
neg_qty  = df['Quantity'] < 0
zero_p   = df['Price'] == 0
no_desc  = df['Description'].isna()
survives = df['Customer ID'].notna() & (df['Price'] > 0)   # строка переживёт очистку

problems = {
    'Пропуск Customer ID': no_id,
    'Quantity < 0':        neg_qty,
    'Price = 0':           zero_p,
    'Пропуск Description':  no_desc,
}

decision = {
    'Пропуск Customer ID': 'Удаляем — без клиента нет RFM',
    'Quantity < 0':        'Оставляем возвраты (снижают Monetary)',
    'Price = 0':           'Удаляем — нулевой вклад в Monetary',
    'Пропуск Description':  'Игнорируем — все и так удаляются',
}

# идём по проблемам по одной и собираем строки таблицы
rows = []
for name, mask in problems.items():
    rows.append({
        'Проблема':        name,
        'Строк':           mask.sum(),
        'Доля, %':         round(mask.mean() * 100, 2),
        'без Customer ID': (mask & no_id).sum(),
        'Price = 0':       (mask & zero_p).sum(),
        'нет Description':  (mask & no_desc).sum(),
        'доживёт до RFM':  (mask & survives).sum(),
        'Решение':         decision[name],
    })

summary = pd.DataFrame(rows).set_index('Проблема')
summary

In [ ]:
# наглядно: доля проблемных строк по типам
plt.figure(figsize=(8, 4))
sns.barplot(x=summary['Доля, %'], y=summary.index, hue=summary.index, palette='flare', legend=False)
plt.xlabel('Доля от всех строк, %')
plt.ylabel('')
plt.title('Доля проблемных строк по типам')
plt.tight_layout()
plt.show()

In [ ]:
# Проверяем, все ли отрицательные Quantity — это возвраты клиентов (Invoice на 'C').
# ВАЖНО: Invoice — смешанный тип (str у отмен, int у обычных). Без astype(str)
# метод .str даёт NaN на числах, и value_counts их прячет — картина искажается.
df[df['Quantity'] < 0]['Invoice'].astype(str).str.startswith('C').value_counts()

## Природа проблем и решения по очистке

Разобрали каждый тип проблемных строк, их пересечения и то, доживает ли
он до расчёта RFM.

**Пропуски в Customer ID — 107 927 (20.5 %).**
Транзакция есть, но клиент не идентифицирован. Для RFM это критично:
метрики считаются на клиента, без ID покупку не к кому привязать.
→ **Удаляем.**

**Отрицательный Quantity — 12 326 (2.3 %).** Делим по наличию Customer ID:
- **9 839** — возвраты клиентов (есть Customer ID, Invoice на «C») — **оставляем**;
- **2 487** — строки без Customer ID: из них 2 121 складские корректировки
  (числовой Invoice, Price = 0) и 366 анонимных возвратов (Invoice на «C»).
  Все 2 487 удаляются вместе с пропусками Customer ID.

Возвраты клиентов оставляем: их отрицательный вклад в Monetary
(`Price × Quantity < 0`) корректно снижает реальную ценность клиента.
→ **Возвраты клиентов оставляем, строки без ID уходят сами.**

*(Ячейка выше делит те же 12 326 иначе — по типу Invoice: 10 205 с «C» и
2 121 числовых. Это деление по типу счёта, а не по наличию клиента,
поэтому числа не совпадают с 9 839 / 2 487.)*

**Нулевая цена — Price = 0 — 3 687 (0.7 %).**
Отдельный признак, не подмножество возвратов: бывает и при `Quantity > 0`
(бесплатные/промо-позиции, не проставленная цена), и при `Quantity < 0`.
Общее одно: вклад в Monetary = 0. Плюс 3 656 из 3 687 и так без клиента.
→ **Удаляем** (причина — нулевой денежный вклад, а не «возврат»).

**Пропуски в Description — 2 928 (0.6 %).**
Все 2 928 одновременно имеют Price = 0 и пустой Customer ID — то есть
пропуск описания встречается только у служебных строк, не у реальных
продаж. Отдельная обработка не нужна: все удаляются фильтрами выше
(доживает до RFM — ноль).
→ **Игнорируем** (Description в RFM не используется и уходит сам).

---

**Про пересечения.** Эти признаки не независимы, а сильно
**перекрываются**: одна служебная строка часто битая сразу по нескольким
причинам (нет Customer ID + Price = 0 + Quantity < 0 + нет Description).
Поэтому нельзя складывать счётчики проблем — суммарно удаляется меньше,
чем `107 927 + 12 326 + 3 687`.

**Итог очистки.** Применяя два фильтра (удалить пропуски Customer ID,
затем удалить Price = 0), из **525 461** строк получаем **417 503**:

- −107 927 — строки без Customer ID;
- −31 — оставшиеся с Price = 0 (всего таких 3 687, но 3 656 уже ушли
  вместе с пропусками Customer ID — проблемы пересекаются, поэтому второй
  фильтр убирает лишь 31 «уникальную» строку).

Осознанно сохранено **9 839** возвратов клиентов (Quantity < 0, но с
Customer ID и Price > 0 — колонка «доживёт до RFM» в таблице выше). Они
уменьшают Monetary, отражая реальную ценность клиента. Сама очистка
применяется на этапе 2 (`02_rfm.ipynb`).